In [25]:
import glob,codecs,os,yaml
import pandas as pd
from unidecode import unidecode
from os.path import expanduser
def ding():
    os.system('afplay /System/Library/Sounds/Submarine.aiff')
from IPython.core.display import HTML,display 

## ⚠️
Pour les cas où le genre des noms est marqué explicitement, on utilise *nomGenre=True*

In [26]:
%store -r numerosKalaba typeKalaba
# numerosKalaba=[1]
print numerosKalaba, typeKalaba
nomGenre=False

[1] Kalaba


In [27]:
debug=0
home = expanduser("~")
repertoire=home+"/sDrive/Cours/Bordeaux/L1-LinguistiqueGenerale/00-ProjetKalaba/"
annee=24
if typeKalaba!="Kanonik":
    serie=repertoire+"%d-K%d/"%(annee,numerosKalaba[0])
else:
    serie=repertoire+"%d-Kanoniks/"%(annee,numerosKalaba[0])
    nomsKalabas=[serie+"Kanonik-%02d/"%num for num in numerosKalaba]
print serie

/Users/gilles/sDrive/Cours/Bordeaux/L1-LinguistiqueGenerale/00-ProjetKalaba/24-K1/


In [28]:
nomDeclarationRad="Declarations-Radicaux.tex"
nomDeclarationDec="Declarations-Decoupages.tex"
nomTableauxRad="Tableaux-Gloses.yaml"
nomDeclaration="Declarations.tex"
nomTableaux="Tableaux.yaml"

In [29]:
flexion=pd.read_csv(serie+"Clozes.txt",sep=";",header=None, names=list(range(20)),encoding="utf8").dropna(axis='columns', how='all')
flexion.head()

,0,1,2,3,5,6,7,8,9,10,11,12,13,14,15
0,#\tNOM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,#,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,#,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,filleFSgNom,fille,NOM,zozuʒ,nozozuʒi,no-zozuʒ-i,"NOM, Genre=F, Nombre=Sg, Cas=Nom",Nom-fille.F-Sg,4.0,Nom-,fille,.F,-Sg,NaN,NaN
4,filleFSgAcc,fille,NOM,zozuʒ,kazozuʒi,ka-zozuʒ-i,"NOM, Genre=F, Nombre=Sg, Cas=Acc",Acc-fille.F-Sg,4.0,Acc-,fille,.F,-Sg,NaN,NaN


In [30]:
categories=flexion[2].dropna().unique().tolist()
categories

[u'NOM', u'VER', u'PRO', u'DET', u'ADJ', u'PREP']

bFlex sélectionne les lignes non-vides correspondant à une catégorie

In [31]:
bFlex={c:(flexion[7].notnull()) & (flexion[2]==c.upper()) for c in categories}
flexion[bFlex["NOM"]]

,0,1,2,3,5,6,7,8,9,10,11,12,13,14,15
3,filleFSgNom,fille,NOM,zozuʒ,nozozuʒi,no-zozuʒ-i,"NOM, Genre=F, Nombre=Sg, Cas=Nom",Nom-fille.F-Sg,4.0,Nom-,fille,.F,-Sg,NaN,NaN
4,filleFSgAcc,fille,NOM,zozuʒ,kazozuʒi,ka-zozuʒ-i,"NOM, Genre=F, Nombre=Sg, Cas=Acc",Acc-fille.F-Sg,4.0,Acc-,fille,.F,-Sg,NaN,NaN
5,filleFSgDat,fille,NOM,zozuʒ,dazozuʒi,da-zozuʒ-i,"NOM, Genre=F, Nombre=Sg, Cas=Dat",Dat-fille.F-Sg,4.0,Dat-,fille,.F,-Sg,NaN,NaN
6,filleFSgObl,fille,NOM,zozuʒ,zozuʒni,zozuʒ-n-i,"NOM, Genre=F, Nombre=Sg, Cas=Obl",fille.F-Obl-Sg,4.0,fille,.F,-Obl,-Sg,NaN,NaN
7,filleFDuNom,fille,NOM,zozuʒ,nozozuʒo,no-zozuʒ-o,"NOM, Genre=F, Nombre=Du, Cas=Nom",Nom-fille.F-Du,4.0,Nom-,fille,.F,-Du,NaN,NaN
8,filleFDuAcc,fille,NOM,zozuʒ,kazozuʒo,ka-zozuʒ-o,"NOM, Genre=F, Nombre=Du, Cas=Acc",Acc-fille.F-Du,4.0,Acc-,fille,.F,-Du,NaN,NaN
9,filleFDuDat,fille,NOM,zozuʒ,dazozuʒo,da-zozuʒ-o,"NOM, Genre=F, Nombre=Du, Cas=Dat",Dat-fille.F-Du,4.0,Dat-,fille,.F,-Du,NaN,NaN
10,filleFDuObl,fille,NOM,zozuʒ,zozuʒno,zozuʒ-n-o,"NOM, Genre=F, Nombre=Du, Cas=Obl",fille.F-Obl-Du,4.0,fille,.F,-Obl,-Du,NaN,NaN
11,filleFPlNom,fille,NOM,zozuʒ,nozozuʒa,no-zozuʒ-a,"NOM, Genre=F, Nombre=Pl, Cas=Nom",Nom-fille.F-Pl,4.0,Nom-,fille,.F,-Pl,NaN,NaN
12,filleFPlAcc,fille,NOM,zozuʒ,kazozuʒa,ka-zozuʒ-a,"NOM, Genre=F, Nombre=Pl, Cas=Acc",Acc-fille.F-Pl,4.0,Acc-,fille,.F,-Pl,NaN,NaN


### Analyse de la case
on fait un dict à partir de la case pour éclater en colonne puis faire un pivot_table

In [32]:
def makeParadigme(row):
    result={}
    parts=row["Case"].split(", ")
    for part in parts[1:]:
        attr,val=part.split("=")
        result[attr.strip()]=val
    return result

# Tableaux pour une catégorie

In [33]:
def reindexNombre(tableau):
    result=tableau
    if "Du" in tableau.columns:
        result=tableau.reindex("Sg Du Pl".split(" "),axis=1)
    elif "Pau" in tableau.columns:
        result=tableau.reindex("Sg Pau Pl".split(" "),axis=1)
    elif "Sg" in tableau.columns:
        result=tableau.reindex("Sg Pl".split(" "),axis=1)
    return result

def reindexPers(tableau):
    result=tableau
    if "3Du" in tableau.columns:
        result=tableau.reindex("3Sg 3Du 3Pl".split(" "),axis=1)
    elif "3Pau" in tableau.columns:
        result=tableau.reindex("3Sg 3Pau 3Pl".split(" "),axis=1)
    elif "3Sg" in tableau.columns:
        result=tableau.reindex("3Sg 3Pl".split(" "),axis=1)
    return result

def reindexTableau(tableau):
    return reindexNombre(reindexPers(tableau))

In [34]:
def makeTableaux(cat):
    # sel sélectionne la catégorie cat dans flexion
    # les colonnes 1, 6 et 7 correspondent à la catégorie, la forme et la case
    sel=bFlex[cat]
    catTableaux={}
    catCF={}
    radCF={}
    cfLexemes={}

    # la colonnne 3 contient le radical
    lDF=flexion[sel][[1,6,7]]
    lDF=flexion[sel][[1,3,6,7]]
    lDF.columns=(u"%s Radical Forme Case"%cat).split(" ")
    # on ajoute une colonne avec un dict correspondant à attribut:valeur
    lDF["dCell"]=lDF.apply(makeParadigme,axis=1)
    if lDF.iloc[0]["dCell"]!={}:
        # on éclate le dict attribut:valeur en colonnes
        # on calcule les dimensions du tableau croisé pour pivot_table
        dfCat=pd.concat([lDF.drop(['Case',"dCell"], axis=1), lDF['dCell'].apply(pd.Series)], axis=1)
        setColonnes=set(dfCat.columns.tolist())
        if cat=="NOM" and not nomGenre:
            setColonnes-=set(["Genre"])
        if "Nombre" in setColonnes:
            colonnes="Nombre"
            lignes=setColonnes-set([cat,"Radical","Forme","CF","Nombre"])
        elif "Pers" in setColonnes:
            colonnes="Pers"
            lignes=list(setColonnes-set([cat,"Radical","Forme","CF","Pers"]))
        if debug: print colonnes, lignes
        # on fait la liste des lexèmes de la catégorie cat
        listCat=dfCat[cat].unique().tolist()
        if debug: display(dfCat.head())
        for l in listCat:
            print l
            catL=dfCat[dfCat[cat].str.encode("utf8")==l.encode("utf8")]
            radical=catL["Radical"].values[0]
            if "CF" in dfCat.columns:
                lCF=catL["CF"].values[0]
                if lCF not in cfLexemes:
                    cfLexemes[lCF]=[]
                cfLexemes[lCF].append(l)
                #
                # reindexNombre présente les colonnes concernant le nombre dans l'ordre traditionnel
                # SG < DU/PAU < PL
                #
                if lCF not in catCF:
                    catTableaux[lCF]=pd.pivot_table(catL,index=lignes,columns=colonnes,values="Forme", aggfunc='first')
                    catCF[lCF]=l
                    radCF[lCF]=radical
            elif cat=="NOM" and nomGenre:
                lCF=catL["Genre"].values[0]
                if lCF not in cfLexemes:
                    cfLexemes[lCF]=[]
                cfLexemes[lCF].append(l)
                if lCF not in catCF:
                    catTableaux[lCF]=pd.pivot_table(catL,index=lignes,columns=colonnes,values="Forme", aggfunc='first')
                    catCF[lCF]=l
                    radCF[lCF]=radical
            else:
                catCF[cat]=l
                radCF[cat]=radical
                catTableaux[cat]=pd.pivot_table(catL,index=lignes,columns=colonnes,values="Forme", aggfunc='first')
        for t in catTableaux:   
            if debug: display(HTML("<h1>%s, %s</h1>"%(t,catCF[t])),reindexTableau(catTableaux[t]))
#             print t,catCF[t],radCF[t]
#             print reindexTableau(catTableaux[t]).to_latex(multirow=True)
    return catTableaux,cfLexemes,catCF,radCF

In [35]:
tableaux={}
tableauxLatex=[]
lexemesCF={}
cfLexs={}
cfRads={}
tableauxLatex.append(ur"\subsection*{Tableaux de flexion}")
for cat in "DET PRO NOM ADJ VER".split(" "):
#    print ur"\section{%s}"%cat
    tableauxLatex.append(ur"\subsubsection*{%s}"%cat)
    tableaux[cat],lexemesCF[cat],cfLexs[cat],cfRads[cat]=makeTableaux(cat)
#     print tableaux[cat]
#     print lexemesCF[cat]
    for t in tableaux[cat]:
#        print t
        tableauxLatex.append(u"\\needspace{6\\baselineskip}")
        tableauxLatex.append(u"\\noindent\n%s %s => %s\\\\"%(t,cfLexs[cat][t],cfRads[cat][t]))
        tableauxLatex.append(reindexTableau(tableaux[cat][t]).to_latex(multirow=True)+u" \\\\ \\medskip")
        tableauxLatex.append("")
print "\n".join(tableauxLatex)

DEF
DEM
IND
PRO
fille
chef
infirmière
louve
peau
livre
lever-n
Clemencia
demande
villageoise
coussin
lumière
chasseuse
obscurité
Violette
table
feu
voleuse
sortilège
mère
lit
chatte
vengeance
serpent
Mahira
Nicole
colère
Elphaba
souffrance
autruche
Katisha
Freja
coyote
côté
bras
disparition
maison
soeur
légume
Agathos
femme
place
harmonie
dégât
hurlement
chat
souris
visage
Hoderi
oeil
homme
Kaleb
proximité
Nabil
robe
marais
combattant
corps
chasseur
poisson
croc
lune
excuse
viande
douleur
jour
tornade
démon
raison
gorge
garçon
loup
nuit
lueur
père
thé
jardin
balai
histoire
forêt
manoir
main
chemin
sorcier
lance
combat
mort
village
maître
dernier-n
milieu
œuf
enfant
bûcher
café
ombre
caillou
cuisine
créature
esprit
coup
plaine
matin
construction
parent
sort
crapaud
blessure
chambre
donjon
nouvelle-n
grimoire
légende
héros
vote
gargouille
fruit
villageois
arbre
cri
dragon
couteau
héroïque
sept
furieux
maigre
profond
différent
nouveau
trois
méchant
rouge
blessé
deux
six
terrible
blanc
qua

# Liste des lexèmes par CF

=> à intégrer aux tableaux...

In [36]:
cfLatex=[]
cfLatex.append(ur"\subsection*{Classes flexionnelles}")
for cat in "NOM ADJ VER DET".split(" "):
    classeCat=lexemesCF[cat]
    if classeCat:
        if debug: print cat+" : "
        cfLatex.append(cat+" : ")
        if debug: print ur"\begin{description}"
        cfLatex.append(ur"\begin{description}")
        for c in classeCat:
            if debug: print ur"\item[- %s] %s"%(c,", ".join(sorted(classeCat[c])))
            cfLatex.append(ur"\item[- %s] %s"%(c,", ".join(sorted(classeCat[c]))))
        if debug: print ur"\end{description}"
        cfLatex.append(ur"\end{description}")
if len(cfLatex)<2:
    cfLatex=[]
print "\n".join(cfLatex)

\subsection*{Classes flexionnelles}
VER : 
\begin{description}
\item[- V1] accueillir, acheter, aller, allumer, apporter, approcher, arriver, arrêter, avoir, boire, brûler-med, changer, chasser-act, compter, connaître, donner, détester, enfermer, enlever, entrer, envahir, hurler, inquiéter, interroger, jardiner, lever, montrer, mourir, nourrir, noyer, parler, passer, pendre, planter, pleurer, pleurer-act, prendre, recouvrir, remercier, survoler, tourner-mov, transformer, transpercer, voir, voler, voler-act, voter, être
\item[- V2] attraper, brûler, cacher, chasser-mov, chercher, courir, demander, descendre, disparaître, dormir, découvrir, dévorer, embrasser, excuser, faire, fuir, illustrer, jeter, lancer, manger, manquer-med, nuire, offrir, partir, placer, porter, posséder, pousser, protéger, préparer, présenter, raconter, reconnaître, reconstruire, redevenir, rejoindre, ressusciter, retourner, revenir, rêvasser, sauter, sortir-mov, supporter, surplomber, tomber, vivre, échanger
\end{d

# Liste des noms par genre

- on ajoute une colonne genre en coupant dans la matrice de traits
- on fait la liste des genres
- on fait la liste des noms par genre
- on fabrique la structure LaTeX

In [37]:
genresLatex=[]

genresLatex.append(ur"\subsection*{Genre des noms}")

# Ajout de la colonne genre
flexion["genre"]=flexion[bFlex["NOM"]][7].str.extract("Genre=(.+?),")

# Liste des genres
genres=sorted([g for g in flexion["genre"].unique().tolist() if isinstance(g,unicode)])
genreNoms={}

# Liste des noms par genre
for g in genres:
    genreNoms[g]=flexion[flexion.genre==g][1].unique().tolist()

# LaTeX
if debug: print ur"\begin{description}"
genresLatex.append(ur"\begin{description}")
for g in genres:
    if debug: print ur"\item[- %s]"%g, ", ".join(sorted(genreNoms[g]))
    genresLatex.append(ur"\item[- %s] %s"%(g, ", ".join(sorted(genreNoms[g]))))
if debug: print ur"\end{description}"
genresLatex.append(ur"\end{description}")

print "\n".join(genresLatex)

\subsection*{Genre des noms}
\begin{description}
\item[- F] Agathos, Clemencia, Elphaba, Freja, Katisha, Mahira, Nicole, Violette, autruche, bras, chasseuse, chatte, chef, colère, coussin, coyote, côté, demande, disparition, femme, feu, fille, infirmière, lever-n, lit, livre, louve, lumière, légume, maison, mère, obscurité, peau, place, serpent, soeur, sortilège, souffrance, table, vengeance, villageoise, voleuse
\item[- M] Hoderi, Kaleb, Nabil, balai, chasseur, chat, combattant, corps, croc, douleur, dégât, démon, excuse, forêt, garçon, gorge, harmonie, histoire, homme, hurlement, jardin, jour, loup, lueur, lune, marais, nuit, oeil, poisson, proximité, père, raison, robe, souris, thé, tornade, viande, visage
\item[- N] arbre, blessure, bûcher, café, caillou, chambre, chemin, combat, construction, coup, couteau, crapaud, cri, créature, cuisine, dernier-n, donjon, dragon, enfant, esprit, fruit, gargouille, grimoire, héros, lance, loup, légende, main, manoir, matin, maître, milieu, mort,

# Calcul des remplissages de paradigmes
- filledCells contient les informations exportées par Phrases2
- sortLevel permet de mettre les noms des traits dans l'ordre logique
- filledTable fait le tableau du remplissage pour une catégorie avec un pivot_table
    - on lui passe 
        - le filledCells de la catégorie
        - la colonne du lexème
        - les colonnes qu'on veut mettre en ligne dans le tableau du paradigme
        - les colonnes qu'on veut mettre en colonnes dans le tableau du paradigme

In [38]:
with open(serie+"FilledCells.yaml", 'r') as stream:
    filledCells=yaml.safe_load(stream)
filledCells["VER"]=[v.replace(".Trois",".3") for v in filledCells["VER"]]
filledCells

{'ADJ': [u'diff\xe9rent.M.SG',
  'noir.M.SG',
  'maigre.F.PL',
  'autre.F.PL',
  'terrible.M.PL',
  'petit.N.PL',
  'gros.M.SG',
  'furieux.N.PL',
  'quatre.M.PL',
  'profond.M.SG',
  'jaune.N.DU',
  'courageux.F.PL',
  'maigre.M.PL',
  'noir.M.PL',
  'jaune.M.DU',
  u'bless\xe9.M.SG',
  'jaune.F.DU',
  'petit.N.SG',
  u'diff\xe9rent.N.SG',
  'gros.F.DU',
  'courageux.M.PL',
  'trois.M.PL',
  'blanc.N.DU',
  'blanc.F.DU',
  'gros.F.SG',
  'maigre.M.SG',
  'noir.F.SG',
  'profond.F.SG',
  'courageux.F.SG',
  'grand.N.SG',
  'petit.F.SG',
  'suivant.N.SG',
  u'effray\xe9.F.DU',
  u'diff\xe9rent.F.DU',
  'terrible.F.DU',
  'courageux.N.DU',
  'vert.F.SG',
  'grand.M.SG',
  'terrible.N.PL',
  'profond.N.SG',
  'blanc.M.PL',
  'grand.M.DU',
  'terrible.N.SG',
  'noir.N.DU',
  'maigre.F.DU',
  'rouge.N.PL',
  'gros.M.DU',
  'grand.N.DU',
  'grand.F.SG',
  'petit.M.PL',
  'petit.F.DU',
  'nombreux.F.PL',
  'autre.N.PL',
  'petit.F.PL',
  'furieux.M.SG',
  u'bless\xe9.N.PL',
  u'effray\xe9.F.S

In [39]:
def sortLevel(level):
    sortedLevel=level
    name="???"
    if "Erg" in level:
        sortedLevel=[f for f in "Erg Abs Dat Obl".split(" ") if f in level]
        name="Cas"
    elif "Acc" in level or "Nom" in level:
        sortedLevel=[f for f in "Nom Acc Dat Obl".split(" ") if f in level]
        name="Cas"
    elif "3Sg" in level:
        sortedLevel=[f for f in u"3Sg 3Du 3Pau 3Pl".split(" ") if f in level]
        name="Pers"
    elif "Sg" in level or "Pl" in level:
        sortedLevel=[f for f in u"Sg Du Pau Pl".split(" ") if f in level]
        name="Nombre"
    elif "Hum" in level or "Ani" in level:
        sortedLevel=[f for f in "Hum Ani Anim Ina Inan".split(" ") if f in level]
        name="Genre"
    elif "H" in level:
        sortedLevel=[f for f in "H A I".split(" ") if f in level]
        name="Genre"
    elif "M" in level or "F" in level or "N" in level:
        sortedLevel=[f for f in "M F N".split(" ") if f in level]
        name="Genre"
    elif "A" in level and "B" in level:
        sortedLevel=[f for f in "A B C D".split(" ") if f in level]
        name="Genre"
    elif "Def" in level:
        sortedLevel=[f for f in "Def Indef Ind Dem".split(" ") if f in level]
        name="DET"

    elif "Prs" in level:
        print level
        sortedLevel=[f for f in "Prs PRS Pst PST Fut FUT".split(" ") if f in level]
        name="Temps"
    elif "V1" in level or "N1" in level or "A1" in level:
        name="CF"
    elif "Vt" in level:
        name="Trans"
    return sortedLevel, name
        
def filledTable(fc,lexeme,rows,cols):
    filled=[f.split(".") for f in fc]
    filled=[f if "hyper" not in f else [c for c in f if c!="hyper"] for f in filled]
    print [f for f in filled if f and len(f)>5]
    for iF,f in enumerate(filled):
        filled[iF]=[c.capitalize() if c[0] not in "123" else c for c in f]
    dfFilled=pd.DataFrame(filled)
    dfFilled.columns=["C"+str(n) for n in dfFilled.columns.tolist()]
    if lexeme!=0:
        dfFilled["lexeme"]=dfFilled["C0"]
        lLexeme="lexeme"
    else:
        lLexeme="C"+str(lexeme)
    lRows=["C"+str(n) for n in rows]
    lCols=["C"+str(n) for n in cols]
#     display(dfFilled)

    
    result=pd.pivot_table(dfFilled,index=lRows,columns=lCols,values=lLexeme,aggfunc="count",dropna = False,fill_value=0)
#     display(result)
    if isinstance(result.index, pd.MultiIndex):
        levels=result.index.levels
        for iL,level in enumerate(levels):
            print iL, level
            sort,name=sortLevel(level)
            result=result.reindex(sort,level=iL)
            result.index=result.index.set_names(name, level=iL)
    else:
        sort,name=sortLevel(result.index)
        result=result.reindex(sort)
        result.index=result.index.set_names(name)

    if isinstance(result.columns, pd.MultiIndex):
        levels=result.columns.levels
        for iL,level in enumerate(levels):
            sort,name=sortLevel(level)
            result=result.reindex(sort,level=iL,axis=1)
            result.columns=result.columns.set_names(name, level=iL)
#             result=result.reindex(sortLevel(level),level=-iL,axis=1)
    else:
        sort,name=sortLevel(result.columns)
        result=result.reindex(sort,axis=1)
        result.columns=result.columns.set_names(name)

#         result=result.reindex(sortLevel(result.columns),axis=1)


    
    return result

In [40]:
tableauxRemplissage={}

# Paramétrage des remplissages

In [41]:
fc=filledCells["NOM"]
print fc[0]
tableauxRemplissage["NOM"]=filledTable(fc,0,[1],[2])
# tableauxRemplissage["NOM"]=filledTable(fc,0,[1],[3])
tableauxRemplissage["NOM"]=filledTable(fc,0,[2],[3])
# tableauxRemplissage["NOM"]=filledTable(fc,0,[1],[2,3])
# tableauxRemplissage["NOM"]=filledTable(fc,0,[1],[3,4])
# tableauxRemplissage["NOM"]=filledTable(fc,0,[2,1],[3,4])
# tableauxRemplissage["NOM"]=filledTable(fc,0,[2],[3,4])


balai.M.Pl.Nom
[]
[]


In [42]:
fc=filledCells["VER"]
print fc[0]
tableauxRemplissage["VER"]=filledTable(fc,0,[1],[2,3])
tableauxRemplissage["VER"]=filledTable(fc,0,[1,2],[3,4])
# tableauxRemplissage["VER"]=filledTable(fc,0,[2,3],[4,5])


lancer.V2.Pst.3Sg.M
[]
Index([u'Prs', u'Pst'], dtype='object', name=u'C2')
[]
0 Index([u'V1', u'V2'], dtype='object', name=u'C1')
1 Index([u'Prs', u'Pst'], dtype='object', name=u'C2')
Index([u'Prs', u'Pst'], dtype='object', name=u'C2')


In [43]:
fc=filledCells["ADJ"]
print fc[0]
tableauxRemplissage["ADJ"]=filledTable(fc,0,[1],[2])
# tableauxRemplissage["ADJ"]=filledTable(fc,0,[1],[2,3])
# tableauxRemplissage["ADJ"]=filledTable(fc,0,[1,2],[3,4])

différent.M.SG
[]


In [44]:
fc=filledCells["DET"]
print fc[0]
tableauxRemplissage["DET"]=filledTable(fc,0,[1],[2])
tableauxRemplissage["DET"]=filledTable(fc,0,[1],[2,3])
# tableauxRemplissage["DET"]=filledTable(fc,0,[1,2],[3,4])

DEF.N.SG.Nom
[]
[]


In [45]:
fc=filledCells["PRO"]
print fc[0]
if fc[0]:
    tableauxRemplissage["PRO"]=filledTable(fc,0,[1],[2])
    tableauxRemplissage["PRO"]=filledTable(fc,0,[1],[2,3])

PRO.M.Pl.Nom
[]
[]


# Tableaux remplis

In [46]:
tableauxLatexRemplissage=[]
tableauxLatexRemplissage.append(ur"\subsection*{Tableaux de remplissage}")
for cat in "DET PRO NOM ADJ VER".split(" "):
#    print ur"\section{%s}"%cat
    if cat in tableauxRemplissage:
        tableauxLatexRemplissage.append(u"%%\\columnbreak\n\\subsubsection*{%s}"%cat)
        tableauxLatexRemplissage.append(tableauxRemplissage[cat].to_latex(multirow=True)+u" \\\\ \\medskip")
        tableauxLatexRemplissage.append("")
print "\n".join(tableauxLatexRemplissage)

\subsection*{Tableaux de remplissage}
%\columnbreak
\subsubsection*{DET}
\begin{tabular}{lrrrrrrrrrrrr}
\toprule
Nombre & \multicolumn{4}{l}{Sg} & \multicolumn{4}{l}{Du} & \multicolumn{4}{l}{Pl} \\
Cas & Nom & Acc & Dat & Obl & Nom & Acc & Dat & Obl & Nom & Acc & Dat & Obl \\
Genre &     &     &     &     &     &     &     &     &     &     &     &     \\
\midrule
M     &   3 &   3 &   3 &   3 &   3 &   3 &   1 &   3 &   3 &   3 &   3 &   2 \\
F     &   3 &   3 &   2 &   3 &   3 &   3 &   3 &   3 &   3 &   2 &   2 &   3 \\
N     &   3 &   3 &   1 &   3 &   3 &   3 &   2 &   3 &   3 &   3 &   2 &   3 \\
\bottomrule
\end{tabular}
 \\ \medskip

%\columnbreak
\subsubsection*{PRO}
\begin{tabular}{lrrr}
\toprule
Nombre &  Sg &  Du &  Pl \\
Cas & Nom & Nom & Nom \\
Genre &     &     &     \\
\midrule
M     &   1 &   0 &   1 \\
F     &   1 &   0 &   1 \\
N     &   0 &   1 &   1 \\
\bottomrule
\end{tabular}
 \\ \medskip

%\columnbreak
\subsubsection*{NOM}
\begin{tabular}{lrrrr}
\toprule
Cas &  

In [47]:
with codecs.open(serie+"Flexions.tex","w",encoding="utf8") as outFile:
    print "\n".join(tableauxLatex+cfLatex+genresLatex+tableauxLatexRemplissage)
    outFile.write("\n".join(tableauxLatex+cfLatex+genresLatex+tableauxLatexRemplissage))

\subsection*{Tableaux de flexion}
\subsubsection*{DET}
\needspace{6\baselineskip}
\noindent
DET IND => la\\
\begin{tabular}{lllll}
\toprule
    & Nombre &        Sg &        Du &        Pl \\
Cas & Genre &           &           &           \\
\midrule
\multirow{3}{*}{Acc} & F &  la-ʃ-a-s &  la-ʃ-a-d &  la-ʃ-a-θ \\
    & M &  la-s-a-s &  la-s-a-d &  la-s-a-θ \\
    & N &  la-ʒ-a-s &  la-ʒ-a-d &  la-ʒ-a-θ \\
\cline{1-5}
\multirow{3}{*}{Dat} & F &  la-ʃ-e-s &  la-ʃ-e-d &  la-ʃ-e-θ \\
    & M &  la-s-e-s &  la-s-e-d &  la-s-e-θ \\
    & N &  la-ʒ-e-s &  la-ʒ-e-d &  la-ʒ-e-θ \\
\cline{1-5}
\multirow{3}{*}{Nom} & F &  la-ʃ-o-s &  la-ʃ-o-d &  la-ʃ-o-θ \\
    & M &  la-s-o-s &  la-s-o-d &  la-s-o-θ \\
    & N &  la-ʒ-o-s &  la-ʒ-o-d &  la-ʒ-o-θ \\
\cline{1-5}
\multirow{3}{*}{Obl} & F &  la-ʃ-i-s &  la-ʃ-i-d &  la-ʃ-i-θ \\
    & M &  la-s-i-s &  la-s-i-d &  la-s-i-θ \\
    & N &  la-ʒ-i-s &  la-ʒ-i-d &  la-ʒ-i-θ \\
\bottomrule
\end{tabular}
 \\ \medskip

\subsubsection*{PRO}
\needspace{6\baseli

In [48]:
ding()